In [29]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [42]:
# Load dataset
bgg_dataset = pd.read_csv("bgg_dataset.csv", sep=';')
bgg_dataset.head()

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,"8,79",1,"3,86",68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,"8,61",2,"2,84",65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
2,224517.0,Brass: Birmingham,2018.0,2,4,120,14,19217,"8,66",3,"3,91",28785.0,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
3,167791.0,Terraforming Mars,2016.0,1,5,120,12,64864,"8,43",4,"3,24",87099.0,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
4,233078.0,Twilight Imperium: Fourth Edition,2017.0,3,6,480,14,13468,"8,70",5,"4,22",16831.0,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"


In [34]:
# Load dataset
df = pd.read_csv("products_with_edition.csv")
df.head()

,productid,objecttype,objectid,producthref,price,price_in_usd,currency,currencystring,currencysymbol,condition,...,quantity,userId,username,user_country,user_city,version_name,version_href,game_name,game_href,linkeduserGeekMarket
0,3870857,thing,1,/market/product/3870857,30.00,30.00,USD,U.S. dollars,$,likenew,...,1,2954044,LeBao,United States,Tallahassee,Die Macher (Multilingual edition 2006),/boardgameversion/24534/multilingual-edition-2006,Die Macher,/boardgame/1/die-macher,NaN
1,3866054,thing,1,/market/product/3866054,180.00,180.00,USD,U.S. dollars,$,new,...,1,3677448,CarsonChandler,United States,Christiansburg,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
2,3854476,thing,1,/market/product/3854476,41.99,41.99,USD,U.S. dollars,$,likenew,...,1,215794,Eric Engelmann,United States,Rockville,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
3,3853310,thing,1,/market/product/3853310,60.00,66.00,EUR,Euros,€,new,...,1,1613268,JillySilly,Belgium,NaN,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
4,3845442,thing,1,/market/product/3845442,50.00,50.00,USD,U.S. dollars,$,verygood,...,1,38240,caesarmom,United States,Lorton,Die Macher (German-only second edition),/boardgameversion/24939/german-only-second-edi...,Die Macher,/boardgame/1/die-macher,NaN


In [36]:
import pandas as pd

# --- 1. Keep only the required columns ---
cols_to_keep = [
    "objectid",
    "price_in_usd",
    "version_name",
    "condition"
]

products_clean = df.loc[:, cols_to_keep].copy()

# --- 2. Drop rows with missing IDs or prices ---
products_clean = products_clean.dropna(subset=["objectid", "price_in_usd"])

# --- 3. Convert data types safely ---
products_clean["objectid"] = pd.to_numeric(
    products_clean["objectid"], errors="coerce"
)
products_clean["price_in_usd"] = pd.to_numeric(
    products_clean["price_in_usd"], errors="coerce"
)

# Drop rows that failed conversion
products_clean = products_clean.dropna(subset=["objectid", "price_in_usd"])
products_clean["objectid"] = products_clean["objectid"].astype(int)

# --- 4. Clean and standardize condition ---
products_clean["condition"] = (
    products_clean["condition"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# --- 5. Keep ONLY 'new' condition ---
products_clean = products_clean[products_clean["condition"] == "new"].copy()

# --- 6. Clean version names ---
products_clean["version_name"] = (
    products_clean["version_name"]
    .astype(str)
    .str.strip()
)

# --- 7. Remove exact duplicate rows ---
products_clean = products_clean.drop_duplicates(
    subset=["objectid", "version_name", "condition", "price_in_usd"]
)

# --- 8. Average duplicated game versions ---
products_clean_avg = products_clean.groupby(
    ["objectid", "version_name", "condition"], as_index=False
)["price_in_usd"].mean()

# ✅ Final verified clean dataset
products_clean_avg.head()


,objectid,version_name,condition,price_in_usd
0,1,Die Macher (German-only second edition),new,73.630000
1,1,Die Macher (Multilingual edition 2006),new,44.500000
2,1,Die Macher (Spanish edition),new,46.200000
3,1,Die Macher (Spielworxx English/German edition ...,new,71.500000
4,1,Die Macher (Spielworxx English/German edition),new,121.998333


In [37]:
# 1️⃣ Check dataset shape
print("Dataset shape:", products_clean_avg.shape)

# 2️⃣ Check for missing values
print("\nMissing values per column:")
print(products_clean_avg.isna().sum())

# 3️⃣ Check that ONLY 'new' condition exists
print("\nUnique conditions:")
print(products_clean_avg["condition"].unique())

# 4️⃣ Check data types
print("\nData types:")
print(products_clean_avg.dtypes)

# 5️⃣ Check for duplicate (objectid + version_name)
duplicates = products_clean_avg.duplicated(
    subset=["objectid", "version_name"]
).sum()

print("\nDuplicate (s duplicated game+version rows:", duplicates)

# 6️⃣ Check for invalid prices (zero or negative)
invalid_prices = (products_clean_avg["price_in_usd"] <= 0).sum()
print("\nInvalid prices (<=0):", invalid_prices)

# 7️⃣ Price summary (sanity check)
print("\nPrice distribution:")
print(products_clean_avg["price_in_usd"].describe())

# 8️⃣ Sample random rows for manual verification
print("\nRandom sample:")
products_clean_avg.sample(5)


Dataset shape: (25352, 4)

Missing values per column:
objectid        0
version_name    0
condition       0
price_in_usd    0
dtype: int64

Unique conditions:
['new']

Data types:
objectid          int64
version_name     object
condition        object
price_in_usd    float64
dtype: object

Duplicate (s duplicated game+version rows: 0

Invalid prices (<=0): 0

Price distribution:
count    25352.000000
mean        53.322664
std        565.629570
min          0.750000
25%         18.150000
50%         30.000000
75%         52.198500
max      80246.400000
Name: price_in_usd, dtype: float64

Random sample:


,objectid,version_name,condition,price_in_usd
24058,287638,Roméo & Juliette (French edition),new,10.890000
11562,124393,Cavemen Playing With Fire (Dutch/English/Germa...,new,11.000000
13602,154098,1914: Germany at War (English edition),new,76.976667
21533,253608,nan,new,90.000000
18896,219278,nan,new,25.850000


In [38]:
#IQR-based filter to get a modeling-safe dataset:
Q1 = products_clean_avg["price_in_usd"].quantile(0.25)
Q3 = products_clean_avg["price_in_usd"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

products_model_ready = products_clean_avg[
    (products_clean_avg["price_in_usd"] >= lower_bound) &
    (products_clean_avg["price_in_usd"] <= upper_bound)
]

print("Shape after outlier removal:", products_model_ready.shape)
print(products_model_ready["price_in_usd"].describe())


Shape after outlier removal: (23347, 4)
count    23347.000000
mean        33.825245
std         21.901633
min          0.750000
25%         17.000000
50%         27.750000
75%         45.000000
max        103.255000
Name: price_in_usd, dtype: float64


In [47]:
# --- Clean numeric columns in BGG dataset ---
numeric_cols = ['Year Published', 'Rating Average', 'Complexity Average', 'BGG Rank']
for col in numeric_cols:
    bgg_dataset[col] = pd.to_numeric(bgg_dataset[col].astype(str).str.replace(',', '.'), errors='coerce')

int_cols = ['Min Players', 'Max Players', 'Play Time', 'Min Age', 'Users Rated', 'Owned Users']
for col in int_cols:
    bgg_dataset[col] = pd.to_numeric(bgg_dataset[col], errors='coerce')

# Strip text columns
for col in ['Name', 'Mechanics', 'Domains']:
    bgg_dataset[col] = bgg_dataset[col].astype(str).str.strip()

# Drop rows with missing values
bgg_cleaned = bgg_dataset.dropna().copy()
print(f"✅ BGG dataset cleaned: {bgg_cleaned.shape[0]} rows remaining.")

# Match IDs between datasets
bgg_cleaned['ID'] = pd.to_numeric(bgg_cleaned['ID'], errors='coerce')
products_model_ready['objectid'] = pd.to_numeric(products_model_ready['objectid'], errors='coerce')
products_model_ready = products_model_ready[products_model_ready['objectid'].isin(bgg_cleaned['ID'])]

# Aggregate by version
products_grouped = (
    products_model_ready.groupby(['objectid', 'version_name'], as_index=False)
    .agg({'price_in_usd': 'mean', 'condition': 'first'})
)

# --- Merge with BGG data ---
bgg_merged = bgg_cleaned.merge(
    products_grouped[['objectid', 'price_in_usd', 'condition', 'version_name']],
    how='left',
    left_on='ID',
    right_on='objectid'
)
bgg_merged.drop(columns=['objectid'], inplace=True)

# --- Clean unrealistic values ---
bgg_merged = bgg_merged[
    (bgg_merged['Year Published'] >= 1900) &
    (bgg_merged['Min Players'] > 0) &
    (bgg_merged['Max Players'] <= 20) &
    (bgg_merged['Play Time'] <= 600) &
    (bgg_merged['Min Age'] >= 0) &
    ((bgg_merged['price_in_usd'].isna()) | (bgg_merged['price_in_usd'] <= 300))
]

print(f"✅ Final cleaned dataset: {bgg_merged.shape}")


✅ BGG dataset cleaned: 20320 rows remaining.


C:\Users\Abdel\AppData\Local\Temp\ipykernel_6024\2803471389.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  products_model_ready['objectid'] = pd.to_numeric(products_model_ready['objectid'], errors='coerce')


✅ Final cleaned dataset: (30980, 17)


In [48]:
# --- Quick checks on merged dataset ---

print("✅ Columns in merged dataset:")
print(bgg_merged.columns)

print("\n✅ Dataset shape:")
print(bgg_merged.shape)

print("\n✅ Missing values per column:")
print(bgg_merged.isna().sum())

print("\n✅ Unique conditions in merged dataset:")
print(bgg_merged['condition'].unique() if 'condition' in bgg_merged.columns else "No condition column")

print("\n✅ Price statistics:")
print(bgg_merged['price_in_usd'].describe())

print("\n✅ Random sample rows:")
print(bgg_merged[['ID', 'Name', 'price_in_usd', 'version_name']].sample(5))


✅ Columns in merged dataset:
Index(['ID', 'Name', 'Year Published', 'Min Players', 'Max Players',
       'Play Time', 'Min Age', 'Users Rated', 'Rating Average', 'BGG Rank',
       'Complexity Average', 'Owned Users', 'Mechanics', 'Domains',
       'price_in_usd', 'condition', 'version_name'],
      dtype='object')

✅ Dataset shape:
(30980, 17)

✅ Missing values per column:
ID                       0
Name                     0
Year Published           0
Min Players              0
Max Players              0
Play Time                0
Min Age                  0
Users Rated              0
Rating Average           0
BGG Rank                 0
Complexity Average       0
Owned Users              0
Mechanics                0
Domains                  0
price_in_usd          8154
condition             8154
version_name          8154
dtype: int64

✅ Unique conditions in merged dataset:
['new' nan]

✅ Price statistics:
count    22826.000000
mean        33.816757
std         21.786933
min         

In [49]:
bgg_final = bgg_merged.dropna(subset=['price_in_usd', 'version_name', 'condition'])
print(f"✅ Final dataset shape: {bgg_final.shape}")


✅ Final dataset shape: (22826, 17)


In [50]:
# Save the cleaned dataset to CSV
bgg_final.to_csv("bgg_final.csv", index=False)

print("✅ bgg_final saved successfully as 'bgg_final.csv'")


✅ bgg_final saved successfully as 'bgg_final.csv'


In [51]:
# Save the cleaned dataset to CSV
bgg_final.to_csv("products_model_ready", index=False)

print("✅ bgg_final saved successfully as 'bgg_final.csv'")


✅ bgg_final saved successfully as 'bgg_final.csv'
